In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

import warnings
warnings.filterwarnings('ignore')

In [2]:
df_cleaned = pd.read_csv("../../../data/insurance.csv")

X = df_cleaned.drop('charges', axis=1)
y = df_cleaned['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Test: {X_test.shape}")

Kích thước tập Train: (1070, 6)
Kích thước tập Test: (268, 6)


In [3]:
def get_preprocessor():
    numeric_features = ['age', 'bmi', 'children']
    categorical_features = ['sex', 'smoker', 'region']

    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])
    
    return preprocessor

preprocessor = get_preprocessor()

In [4]:
svr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', SVR())
])

param_grid = {
    'model__kernel': ['linear', 'rbf'],
    'model__C': [0.1, 10, 100, 1000, 5000], 
    'model__gamma': ['scale', 'auto', 0.1]
}

grid_search = GridSearchCV(
    estimator=svr_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1, 
    verbose=2
)

In [5]:
grid_search.fit(X_train, y_train)

best_svr_model = grid_search.best_estimator_

y_pred = best_svr_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n" + "=================================")
print("SUPPORT VECTOR REGRESSOR")
print("=================================")
print(f"Tham số tối ưu : {grid_search.best_params_}")
print(f"Test MAE      : {mae:,.2f}")
print(f"Test RMSE     : {rmse:,.2f}")
print(f"Test R² Score : {r2:.4f}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits

SUPPORT VECTOR REGRESSOR
Tham số tối ưu : {'model__C': 5000, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
Test MAE      : 2,018.45
Test RMSE     : 5,284.87
Test R² Score : 0.8201


In [6]:
import wandb

run = wandb.init(
    project="Medical-Insurance-Cost-Prediction",  
    name="SupportVectorRegressor",
    config={
        "model": "SupportVectorRegressor",
        "C": 1.0,
        "cv_folds": 5,
        "max_iter": 1000,
        "test_size": 0.2,
        "random_state": 42
    }
)

print(run.config)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: ngocdo5852 (ngocdo5852-dai-hoc-mo). Use `wandb login --relogin` to force relogin


{'model': 'SupportVectorRegressor', 'C': 1.0, 'cv_folds': 5, 'max_iter': 1000, 'test_size': 0.2, 'random_state': 42}


In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred, alpha=0.5, color='blue')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2) # Đường chuẩn y=x
ax.set_xlabel('Chi phí thực tế (Actual)')
ax.set_ylabel('Chi phí dự đoán (Predicted)')
ax.set_title('Biểu đồ Thực tế vs Dự đoán - SVR')

wandb.log({
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2,
    "Actual_vs_Predicted": wandb.Image(fig)
})

plt.close(fig)